blalba

In [27]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

In [50]:
def save_plot(samples, target_func, accepted_samples, rejected_samples, total_samples, image_number):

    plt.figure(figsize=(12, 6))

    # Trace Plot: Accepted and Rejected Samples
    plt.subplot(1, 2, 1)
    accepted_x, accepted_y = zip(*accepted_samples) if accepted_samples else ([], [])
    rejected_x, rejected_y = zip(*rejected_samples) if rejected_samples else ([], [])

    plt.scatter(accepted_y, accepted_x, color='lightseagreen', label='Accepted', s=10)
    plt.scatter(rejected_y, rejected_x, color='indianred', label='Rejected', s=10)
    plt.xlabel('Sample Value', fontsize=14)
    plt.ylabel('Iteration', fontsize=14)
    plt.title('Trace Plot of MCMC Samples', fontsize=14)
    plt.ylim(0, total_samples)

    # Samples Overview
    plt.subplot(1, 2, 2)
    labels = ['Accepted Samples', 'Rejected Samples']
    counts = [len(accepted_samples), len(rejected_samples)]

    total_counts = len(accepted_samples) + len(rejected_samples)
    proportions = [len(accepted_samples) / total_counts, len(rejected_samples) / total_counts] if total_counts > 0 else [0, 0]

    bar_width = 0.4
    x = np.arange(len(labels))

    plt.bar(x, proportions, color=['lightseagreen', 'indianred'], width=bar_width)
    plt.ylabel('Proportion', fontsize=14)
    plt.title(f'Total Attempts: {total_samples}', fontsize=14)
    plt.ylim(0, 1)

    for index, value in enumerate(proportions):
        plt.text(index, value + 0.01, f"{value:.2f}", ha='center', va='bottom', fontsize=12)

    plt.xticks(x, labels, fontsize=12)

    plt.tight_layout()

    # Save the figure
    plt.savefig(f'accepted_samples_mcmc_{image_number:03d}.png')
    plt.close()

In [51]:
def target_distribution(state):
    if state == 'R':
        return 0.3  # probability for Rainy
    elif state == 'S':
        return 0.7  # probability for Sunny
    return 0

def proposal_distribution(current_state,symm=True):
    if symm == True:
      return np.random.choice(['R', 'S'])
    else:
      if current_state == 'R':
          return np.random.choice(['R', 'S'], p=[0.7, 0.3])
      elif current_state == 'S':
          return np.random.choice(['R', 'S'], p=[0.4, 0.6])

def proposal_probability(proposed_state, current_state, symm=True):
  if symm == True:
    return 1
  else:
    if current_state == 'R':
        return 0.7 if proposed_state == 'R' else 0.3
    elif current_state == 'S':
        return 0.4 if proposed_state == 'R' else 0.6
    return 0

# mcmc + metropolis hasting algorithm
def mcmc(n_samples, initial_state='S', symm=True, save_interval=100,max_images=10): # symm=True for symmetric proposal function p(x_n|x') = p(x'|x_n)
    current_state = initial_state
    samples = [current_state]
    accepted_samples = []
    rejected_samples = []
    total_samples = 1
    image_counter = 0

    for i in range(n_samples):
        proposed_state = proposal_distribution(current_state,symm)

        # calculate acceptance ratio
        acceptance_prob = min(1,(target_distribution(proposed_state)*proposal_probability(current_state,proposed_state,symm))/ (target_distribution(current_state)*proposal_probability(proposed_state, current_state,symm)))

        # draw u~U[0,1) and check wether new sample is accepted
        if np.random.random() < acceptance_prob:
          current_state = proposed_state
          # save accepted samples
          accepted_samples.append((total_samples,current_state))
          samples.append(proposed_state)
        else:
          rejected_samples.append((total_samples,current_state))
          samples.append(current_state)
        total_samples += 1

        if total_samples % save_interval == 0 and image_counter < max_images:
              image_counter += 1
              save_plot(samples, target_distribution, accepted_samples, rejected_samples, total_samples, image_counter)

    return samples, accepted_samples, rejected_samples

def analyze_mcmc_results(samples):
    # calculate sample probabilities
    counter = Counter(samples)
    total = len(samples)
    probabilities = {state: count / total for state, count in counter.items()}
    return probabilities

In [52]:
n_samples = 100000
samples, accepted_samples, rejected_samples = mcmc(n_samples)
probabilities = analyze_mcmc_results(samples)

total_samples = len(accepted_samples) + len(rejected_samples)
print(f"Total samples drawn: {total_samples}")
print(f"Accepted samples: {len(accepted_samples)}")
print(f"Rejected samples: {len(rejected_samples)}")
print(f"Probabilities: {probabilities}")

Total samples drawn: 100000
Accepted samples: 80024
Rejected samples: 19976
Probabilities: {'S': 0.7026729732702673, 'R': 0.2973270267297327}


In [53]:
n_samples = 100000
samples, accepted_samples, rejected_samples = mcmc(n_samples,symm=False)
probabilities = analyze_mcmc_results(samples)

total_samples = len(accepted_samples) + len(rejected_samples)
print(f"Total samples drawn: {total_samples}")
print(f"Accepted samples: {len(accepted_samples)}")
print(f"Rejected samples: {len(rejected_samples)}")
print(f"Probabilities: {probabilities}")

Total samples drawn: 100000
Accepted samples: 80832
Rejected samples: 19168
Probabilities: {'S': 0.7016729832701672, 'R': 0.2983270167298327}
